# 18 — Sintéticos IV: aumento de datos (real + sintético)

> **Estado: ESQUELETO.** Este notebook fija el diseño antes de implementar nada. Todas las secciones de trabajo son celdas `TODO` controladas por el flag `IMPLEMENTADO` (por defecto `False`): con `False` el notebook se ejecuta de principio a fin sin error y solo informa de lo pendiente; con `True` cada `TODO` sin implementar levanta `NotImplementedError`. Ninguna cifra de este texto está escrita a mano: las que aparezcan saldrán de celdas.

## 0. Objetivo

Comprobar si entrenar los detectores con **real + sintético** mejora su detección cuando se
evalúan **solo sobre el real OOS**, con el mismo protocolo walk-forward y las mismas métricas
del benchmark ADR-003.

## Preguntas que responde

1. ¿Mejora el aumento la detección OOS real (F1 por evento, cobertura, falsas alarmas) de cada
   detector frente a su versión solo-real?
2. ¿Cómo depende la mejora de la proporción sintético/real (ablación)?
3. ¿Qué generador aporta más y en qué familias de detector?
4. ¿La mejora sobrevive al intervalo de *bootstrap* por bloques y a la comparación con el azar?

## Entradas y salidas (vía `regimenes.rutas`)

| papel | ruta |
|---|---|
| paneles reales | `rutas.DATA_PROCESSED` |
| generadores admitidos y configuración | `rutas.RESULTS_SINTETICOS / 'validacion'`, `rutas.SINTETICOS_CONFIG` |
| trayectorias por pliegue (gitignored) | `rutas.DATA_SINTETICOS / 'aumento'` |
| referencia solo-real (benchmark) | `rutas.RESULTS_BENCHMARK` |
| resultados del aumento | `rutas.RESULTS_SINTETICOS / 'aumento'` |

## Metodología prevista

- **Generador por pliegue.** En cada re-ajuste del walk-forward el generador se re-entrena **solo
  con el tramo de entrenamiento de ese pliegue**; las trayectorias se generan después y nunca ven
  el bloque que se va a predecir.
- **Entrenamiento aumentado.** El detector se ajusta con el tramo real más trayectorias sintéticas
  (concatenadas como episodios independientes o vía pesos, según admita el detector); la
  predicción y todas las métricas se calculan **exclusivamente sobre el real OOS**.
- **Ablación por proporción.** Rejilla de proporciones sintético/real fijada en la celda de
  configuración (incluye 0 = solo real como control); misma semilla y mismo pliegue.
- **Comparación.** Diferencia pareada frente a solo-real por detector y pista, con IC por
  *bootstrap* por bloques y el nulo aleatorio persistente de `regimenes.evaluacion.ranking`.

## Riesgos

- **Fuga temporal.** Todo generador se ajusta **solo** con el tramo de entrenamiento
  (`entrenamiento.fin_train` de `configs/sinteticos.yaml`); normalizaciones, selección de
  hiperparámetros y etiquetas de régimen de referencia también se calculan solo con ese tramo.
  Mientras `fin_train` sea `null` no se puede pasar `IMPLEMENTADO=True` (la celda de
  configuración lo comprueba).
- **Circularidad.** Si las etiquetas de régimen con las que se condiciona el generador salen de un
  detector del benchmark, ese detector (y su familia) juega en casa al evaluarse sobre lo
  generado. Se mitiga con etiquetas de referencia independientes de los detectores (ventanas de
  crisis y suelos de drawdown de `configs/benchmark_spec.yaml`, o varias etiquetas de familias
  distintas) y leyendo la matriz generador × detector con la diagonal "misma familia" aparte.
- **Fuga por el generador.** Un generador ajustado con toda la muestra transmitiría el futuro al
  entrenamiento aumentado: por eso se re-entrena por pliegue (coste alto, asumido).
- **Selección post-hoc de la proporción.** Elegir la mejor proporción mirando el OOS es sobreajuste:
  se reporta la curva entera y la elección se difiere a `19` con criterio fijado de antemano.

**Índice**

0. Objetivo, preguntas, entradas/salidas, metodología y riesgos
1. Configuración
2. Generadores por pliegue walk-forward
3. Entrenamiento aumentado y evaluación en real OOS
4. Ablación por proporción de sintético
5. Comparación con el benchmark real
6. Conclusiones


## 1. Configuración

In [1]:
# Configuración común del esqueleto (no lee results/benchmark ni ajusta nada).
import yaml
import pandas as pd
from IPython.display import display

from regimenes import rutas
from regimenes import sinteticos
from regimenes.sinteticos import Generador
from regimenes.sinteticos import registry as registro_sinteticos

IMPLEMENTADO = False  # pasar a True solo cuando las secciones TODO estén implementadas

CONFIG_SINTETICOS = yaml.safe_load(rutas.SINTETICOS_CONFIG.read_text(encoding='utf-8'))
SEMILLA = CONFIG_SINTETICOS.get('semilla')


def saltar(seccion: str, detalle: str) -> None:
    """Punto TODO controlado: informa y sigue (solo se llama con IMPLEMENTADO=False)."""
    print(f'[TODO pendiente] {seccion}: {detalle}')


def tabla_rutas(rutas_nb: dict) -> pd.DataFrame:
    """Entradas/salidas del notebook (siempre vía regimenes.rutas) y si existen ya en disco."""
    filas = [
        {'papel': papel, 'ruta': ruta.relative_to(rutas.ROOT).as_posix(), 'existe': ruta.exists()}
        for papel, ruta in rutas_nb.items()
    ]
    return pd.DataFrame(filas).set_index('papel')

RUTAS_NB = {
    'entrada: paneles reales': rutas.DATA_PROCESSED,
    'entrada: veredicto de admisión': rutas.RESULTS_SINTETICOS / 'validacion',
    'entrada: configuración sintéticos': rutas.SINTETICOS_CONFIG,
    'salida: trayectorias por pliegue': rutas.DATA_SINTETICOS / 'aumento',
    'referencia: benchmark solo-real (solo lectura)': rutas.RESULTS_BENCHMARK,
    'salida: aumento': rutas.RESULTS_SINTETICOS / 'aumento',
}
display(tabla_rutas(RUTAS_NB))
# Rejilla de proporciones sintético/real (0 = control solo-real). Se fija ANTES de ver resultados.
PROPORCIONES = [0.0]  # TODO: completar la rejilla al implementar
print('Proporciones sintético/real declaradas:', PROPORCIONES)
print('Generadores registrados:', sorted(registro_sinteticos.GENERADORES) or 'ninguno todavía')


,ruta,existe
papel,,
entrada: paneles reales,data/processed,True
entrada: veredicto de admisión,results/sinteticos/validacion,False
entrada: configuración sintéticos,configs/sinteticos.yaml,True
salida: trayectorias por pliegue,data/sinteticos/aumento,False
referencia: benchmark solo-real (solo lectura),results/benchmark,True
salida: aumento,results/sinteticos/aumento,False


Proporciones sintético/real declaradas: [0.0]
Generadores registrados: ninguno todavía


In [2]:
fin_train = CONFIG_SINTETICOS['entrenamiento'].get('fin_train')
if fin_train is None:
    mensaje = ('configs/sinteticos.yaml: entrenamiento.fin_train = null. Fijar el corte de '
               'entrenamiento antes de ajustar cualquier generador (regla anti-fuga).')
    if IMPLEMENTADO:
        raise ValueError(mensaje)
    print('[aviso]', mensaje)
else:
    print('Corte de entrenamiento de los generadores:', fin_train)


[aviso] configs/sinteticos.yaml: entrenamiento.fin_train = null. Fijar el corte de entrenamiento antes de ajustar cualquier generador (regla anti-fuga).


## 2. Generadores por pliegue walk-forward

In [3]:
# TODO 18.2 — generador re-entrenado por pliegue
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. reutilizar los cortes de regimenes.evaluacion.walk_forward (step/train del registro)
    # 2. en cada corte t: gen.fit(panel[:t], regimen_ref[:t]); sample(...) -> rutas.DATA_SINTETICOS / 'aumento'
    raise NotImplementedError('TODO 18.2: re-entrenar generadores por pliegue')
else:
    saltar('18.2', 'generador re-entrenado por pliegue')


[TODO pendiente] 18.2: generador re-entrenado por pliegue


## 3. Entrenamiento aumentado y evaluación en real OOS

In [4]:
# TODO 18.3 — detector aumentado evaluado solo en real
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name
from regimenes import evaluacion as ev

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. detector.fit(real_train + sintético(proporción)); predict solo sobre el bloque real OOS
    # 2. métricas del benchmark (regimenes.evaluacion) sobre el real OOS
    raise NotImplementedError('TODO 18.3: entrenar con aumento y evaluar en real OOS')
else:
    saltar('18.3', 'detector aumentado evaluado solo en real')


[TODO pendiente] 18.3: detector aumentado evaluado solo en real


## 4. Ablación por proporción de sintético

In [5]:
# TODO 18.4 — ablación por proporción
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. curva métrica(proporción) por detector/pista/generador con IC bootstrap por bloques
    raise NotImplementedError('TODO 18.4: ejecutar la ablación por proporción')
else:
    saltar('18.4', 'ablación por proporción')


[TODO pendiente] 18.4: ablación por proporción


## 5. Comparación con el benchmark real

In [6]:
# TODO 18.5 — comparación pareada frente a solo-real
from regimenes.sinteticos import Generador  # interfaz común fit/sample/name

if IMPLEMENTADO:
    # Cuerpo previsto:
    # 1. diferencia pareada vs proporción 0 y vs nulo aleatorio persistente (regimenes.evaluacion.ranking)
    raise NotImplementedError('TODO 18.5: comparar con el benchmark solo-real')
else:
    saltar('18.5', 'comparación pareada frente a solo-real')


[TODO pendiente] 18.5: comparación pareada frente a solo-real


## 6. Conclusiones

Pendiente de implementación (cifras siempre impresas por celda). Siguiente: `19_decision_final`.